**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/qi-2026"], check=True)
    os.chdir("/content/encore/entries/qi-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Qi et al. (2026) from dandiset 001960: intrinsic properties and spontaneous firing

Paper. Qi, Parot, Wang, Pomeroy, Lavis and Cohen, "Large-scale all-optical electrophysiology
links excitability and connectivity to circuit dynamics in vivo", bioRxiv preprint
doi:10.64898/2026.10.05.756595 (version 1); the claims file was written from its full text and
supplement. Dataset. DANDI:001960 version 0.261013.2016, the only published version (see the
first notebook, `qi2026_excitability_connectivity.ipynb`, for the choice and the full data check).

| Claim | Tested here |
|---|---|
| fig5a-rheobase-predicts-spontaneous-excitability, supp22-rheobase-correlations, fig5b-adaptation-dvdt, fig5c-adp-bursting | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import time
import numpy as np
import pandas as pd
from scipy import stats
import l1opto as L

T0 = time.time()
print("dandiset", L.DANDISET, "version", L.VERSION)
U = {(m, "sparseOpto"): L.units(m, "sparseOpto") for m in L.MICE}
PROPS = {m: L.intrinsic_properties(m) for m in L.MICE}
QC_OPTO = {m: U[m, "sparseOpto"]["passed_qc"] for m in L.MICE}
GFP_OPTO = {m: U[m, "sparseOpto"]["gfp_positive"] for m in L.MICE}
iadp, irh, imax, iad, itau = range(5)
matlab_prctile = L.matlab_prctile
for m in L.MICE:
    print(m, "sparseOpto: QC cells", QC_OPTO[m].sum(), "of", len(QC_OPTO[m]), "| property ranges",
          {p: (round(float(np.nanmin(PROPS[m][:, j])), 3), round(float(np.nanmax(PROPS[m][:, j])), 3)) for j, p in enumerate(L.PROPERTIES)})

## Figure 5: intrinsic properties and connectivity in spontaneous activity

Panels A to C of Figure 5 and Supplementary Figure 22 compare each cell's intrinsic properties with
its spontaneous activity in the same sparseOpto session (code, reproduce_figure5_from_nwb.m:
spontaneous spikes are taken in chunks where no QC cell within 80 um received blue). One
streaming pass per mouse (`l1opto.opto_pass`) accumulates the histograms.

In [ ]:
OPTO = {}
for m in L.MICE:
    tl = time.time()
    OPTO[m] = L.opto_pass(m)
    o = OPTO[m]
    q = QC_OPTO[m]
    print(f"{m}: pass {time.time() - tl:.0f} s; blue step levels {np.round(o['levels'], 3)}; median-rule and stored quiet masks agree on "
          f"{100*o['median_vs_stored_quiet_agreement']:.0f}% of frames; QC cells: spontaneous frames per cell median "
          f"{np.median(o['spont_all'].sum(1)[q]):.0f}, ramp frames median {np.median(o['ramp_all'].sum(1)[q]):.0f}, "
          f"quiet spontaneous spikes median {np.median([len(x) for x, k in zip(o['quiet_spikes'], q) if k]):.0f}")

### fig5a-rheobase-predicts-spontaneous-excitability
> Cells with higher optical rheobase fire less at a given subthreshold voltage, both under ramp
> stimulation and during quiet spontaneous activity.

As the code (panel A, M-YQ0201-27): QC cells in five equal-width rheobase bins between the 1st and
99th percentiles (MATLAB prctile; cells outside dropped); P(spk|V) from counts pooled per bin;
the rate at V = 0.5 SH is the bin [0.50, 0.55). Caption quintiles as a variant. Reproduced when the
rate falls across the five bins with Spearman rho below -0.8 in both conditions.

In [ ]:
VE5 = np.linspace(-1, 1.5, 51)
i05 = int(np.searchsorted(VE5, 0.5, side="right") - 1)  # bin [0.50, 0.55)
print("V bin used:", VE5[i05], "to", VE5[i05 + 1])
for m in L.MICE:
    o = OPTO[m]
    q = np.nonzero(QC_OPTO[m])[0]
    rh = PROPS[m][q, irh]
    for scheme in ("code: equal-width bins, 1st to 99th percentile", "caption: quintiles"):
        if scheme.startswith("code"):
            ed = np.linspace(matlab_prctile(rh, 1), matlab_prctile(rh, 99), 6)
        else:
            ed = np.percentile(rh, np.linspace(0, 100, 6)); ed[-1] += 1e-12
        bins_ = np.digitize(rh, ed)  # 1..5 inside
        res = {}
        for cond, (ka, ks) in [("spontaneous", ("spont_all", "spont_spk")), ("ramp", ("ramp_all", "ramp_spk"))]:
            r_ = [o[ks][q[bins_ == k], i05].sum() / o[ka][q[bins_ == k], i05].sum() / L.DT for k in range(1, 6)]
            res[cond] = (np.round(r_, 2), stats.spearmanr(range(5), r_)[0])
        print(f"{m}, {scheme}: n = {np.sum((bins_ >= 1) & (bins_ <= 5))} cells, per bin {[int(np.sum(bins_ == k)) for k in range(1, 6)]}")
        for cond, (r_, rho) in res.items():
            print(f"   {cond}: rate at V = 0.5 by rheobase bin (low to high) {r_} Hz, Spearman rho {rho:.2f}")

**Result: reproduced.** In M-YQ0201-27 with the code's bins (280 cells), the rate at V = 0.5 SH falls
monotonically with rheobase in both conditions (spontaneous 8.9 to 1.4 Hz, ramp 28.6 to 7.2 Hz;
Spearman rho -1.0 for both). Caption quintiles give rho -0.90 and -1.0; M-YQ0201-29 gives -0.90 and
-1.0 with the code's bins.

### supp22-rheobase-correlations
> Optical rheobase is negatively correlated with the conditional spike rate at V = 0.5 SH, both
> evoked (R = -0.59) and spontaneous (R = -0.32).

Per cell, P(spk|V) in the bin [0.50, 0.55) SH, both mice pooled. Readings: (a) QC cells with a finite
rate in both conditions; (b) QC cells inside the 1st to 99th percentile of rheobase in each mouse.

In [ ]:
def per_cell_rate(o, ka, ks, idx):
    with np.errstate(invalid="ignore", divide="ignore"):
        return o[ks][:, idx] / o[ka][:, idx] / L.DT


rows = []
for m in L.MICE:
    o = OPTO[m]
    ev, sp = per_cell_rate(o, "ramp_all", "ramp_spk", i05), per_cell_rate(o, "spont_all", "spont_spk", i05)
    rh = PROPS[m][:, irh]
    q = QC_OPTO[m]
    lo, hi = matlab_prctile(rh[q], 1), matlab_prctile(rh[q], 99)
    rows.append(pd.DataFrame(dict(mouse=m, qc=q, rheobase=rh, evoked=ev, spont=sp, inrange=q & (rh >= lo) & (rh <= hi),
                                  adaptation=PROPS[m][:, iad], vadp=PROPS[m][:, iadp], gfp=GFP_OPTO[m])))
CELL5 = pd.concat(rows, ignore_index=True)
for nm, sel in [("(a) QC, finite in both", CELL5.qc & np.isfinite(CELL5.evoked) & np.isfinite(CELL5.spont)),
                ("(b) QC within 1st-99th percentile rheobase", CELL5.inrange & np.isfinite(CELL5.evoked) & np.isfinite(CELL5.spont))]:
    d = CELL5[sel]
    r1, p1 = stats.pearsonr(d.rheobase, d.evoked)
    r2, p2 = stats.pearsonr(d.rheobase, d.spont)
    print(f"{nm}: n = {len(d)}; evoked R = {r1:.3f} (p = {p1:.2g}); spontaneous R = {r2:.3f} (p = {p2:.2g})")
    for m, dm in d.groupby("mouse"):
        print(f"   {m}: n = {len(dm)}, evoked R = {stats.pearsonr(dm.rheobase, dm.evoked)[0]:.3f}, spontaneous R = {stats.pearsonr(dm.rheobase, dm.spont)[0]:.3f}")
print("QC cells without a finite evoked or spontaneous rate in the bin:", int((CELL5.qc & ~(np.isfinite(CELL5.evoked) & np.isfinite(CELL5.spont))).sum()))

**Result: reproduced.** Over the 543 QC cells of the two sparseOpto sessions (the paper's n = 543),
rheobase correlates with the rate at V = 0.5 SH with R = -0.573 evoked (p = 1.3e-48) and R = -0.311
spontaneous (p = 1.2e-13), within tolerance of -0.59 and -0.32. Reading (b) (531 cells) gives -0.559
and -0.305.

### fig5b-adaptation-dvdt
> Cells with more spike-rate adaptation spike preferentially in response to rapidly rising
> subthreshold voltage during spontaneous activity (R = 0.26 between adaptation and the conditional
> rate at dV/dt = 0.35).

dV/dt is the code's 8-frame centred difference (SH per 8 frames); the rate at dV/dt = 0.35 is the
bin [0.34, 0.37). Methods reading first: dV/dt 5 frames (6 ms) before the spike, V between 0.2 and
0.8 SH, stored quiet mask. Code reading: shift 4 frames, V bins centred inside 0.2 to 0.8 (0.25 to
0.75 SH), median-rule quiet. Cells as in supp22 (reading (a): QC cells with a finite rate). The
quintile curves are the code's: equal-width adaptation bins between the 1st and 99th percentiles,
counts pooled per bin (M-YQ0201-27 as the code; both mice shown).

In [ ]:
DE5 = np.linspace(-0.5, 1, 51)
j35 = int(np.searchsorted(DE5, 0.35, side="right") - 1)
print("dV/dt bin used:", DE5[j35].round(3), "to", DE5[j35 + 1].round(3))
vcj = (np.linspace(-1, 1.5, 11)[:-1] + np.linspace(-1, 1.5, 11)[1:]) / 2
inw = (vcj > 0.2) & (vcj < 0.8)
for nm in ("Methods", "code"):
    vals = []
    for m in L.MICE:
        o = OPTO[m]
        if nm == "Methods":
            num, den = o["jm_spk"][:, j35], o["jm_all"][:, j35]
        else:
            num, den = o["joint_spk"][:, inw, j35].sum(1), o["joint_all"][:, inw, j35].sum(1)
        with np.errstate(invalid="ignore", divide="ignore"):
            vals.append(num / den / L.DT)
    CELL5[f"dvdt_{nm}"] = np.concatenate(vals)
    d = CELL5[CELL5.qc & np.isfinite(CELL5[f"dvdt_{nm}"])]
    r, p = stats.pearsonr(d.adaptation, d[f"dvdt_{nm}"])
    print(f"{nm} reading: n = {len(d)} QC cells with a finite rate, R = {r:.3f}, p = {p:.2g}; per mouse "
          f"{ {mm: round(stats.pearsonr(dm.adaptation, dm[f'dvdt_{nm}'])[0], 3) for mm, dm in d.groupby('mouse')} }")
    for m in L.MICE:
        o = OPTO[m]
        q = np.nonzero(QC_OPTO[m])[0]
        ad = PROPS[m][q, iad]
        ed = np.linspace(matlab_prctile(ad, 1), matlab_prctile(ad, 99), 6)
        bins_ = np.digitize(ad, ed)
        rr = []
        for k in range(1, 6):
            rows_ = q[bins_ == k]
            if nm == "Methods":
                rr.append(o["jm_spk"][rows_, j35].sum() / o["jm_all"][rows_, j35].sum() / L.DT)
            else:
                rr.append(o["joint_spk"][rows_][:, inw, j35].sum() / o["joint_all"][rows_][:, inw, j35].sum() / L.DT)
        print(f"   {m} quintile curves at dV/dt = 0.35, adaptation low to high: {np.round(rr, 2)} Hz; highest bin has the highest rate: {int(np.argmax(rr)) == 4}")

**Result: reproduced.** Under the Methods reading adaptation correlates with the rate at dV/dt = 0.35
with R = 0.274 (p = 8.3e-11, 543 cells), and under the code's with R = 0.262; both within 0.026 of
0.26. In M-YQ0201-27, the session of the code's panel B, the highest-adaptation bin has the highest
rate (12.5 Hz Methods, 13.9 Hz code); in M-YQ0201-29 the fourth bin is highest.

### fig5c-adp-bursting
> Cells with greater after-depolarization fire more bursts during spontaneous activity (R = 0.52
> between V_adp and the 2.5 to 15 ms autocorrelogram).

Burst rate: mean spike autocorrelogram over lags 2.5 to 15 ms (frames 2 to 11, Methods), triggers
the spontaneous no-blue spikes in usable chunks, targets all no-blue spikes. Main reading: triggers
in any state (Results: quiet and whisking); variant: quiet triggers only (code panel C). Cells as in
supp22 (QC cells with a finite rate).

In [ ]:
lagsel = np.arange(140 + 2, 140 + 12)
for nm, key in [("any state (Results)", "any_spikes"), ("quiet only (code)", "quiet_spikes")]:
    br = []
    for m in L.MICE:
        o = OPTO[m]
        b_ = np.full(320, np.nan)
        for c in np.nonzero(QC_OPTO[m])[0]:
            ac = L.self_correlogram(o[key][c], o["no_blue_spikes"][c], o["n_frames"])
            b_[c] = ac[lagsel].mean() / L.DT
        br.append(b_)
    CELL5[f"burst_{nm}"] = np.concatenate(br)
    d = CELL5[CELL5.qc & np.isfinite(CELL5.evoked) & np.isfinite(CELL5.spont) & np.isfinite(CELL5[f"burst_{nm}"])]
    r, p = stats.pearsonr(d.vadp, d[f"burst_{nm}"])
    print(f"{nm}: n = {len(d)}, R = {r:.3f}, p = {p:.2g}; per mouse "
          f"{ {mm: round(stats.pearsonr(dm.vadp, dm[f'burst_{nm}'])[0], 3) for mm, dm in d.groupby('mouse')} }; "
          f"burst rate median {d[f'burst_{nm}'].median():.2f} Hz")

**Result: reproduced.** With spontaneous triggers in any state (Results) V_adp correlates with the 2.5
to 15 ms burst rate with R = 0.546 (p = 1.4e-43, 543 cells), within 0.052 of 0.52. With the code's
quiet-only triggers R = 0.593, outside the tolerance.

## Summary

| Claim | Outcome |
|---|---|
| fig5a-rheobase-predicts-spontaneous-excitability | reproduced |
| supp22-rheobase-correlations | reproduced |
| fig5b-adaptation-dvdt | reproduced |
| fig5c-adp-bursting | reproduced |

How the analysis differs from the paper's. It follows the code's Figure 5 streaming pass on the
stored subthreshold voltage, blue waveforms and spike times. The selection of 543 cells is the QC
cells of the two sparseOpto sessions, which is the paper's n. The rate at V = 0.5 SH is read from
the bin [0.50, 0.55) SH and at dV/dt = 0.35 from [0.34, 0.37).

What was found about the dataset. The median-rule and stored quiet masks agree on 84% of frames
in M-YQ0201-27 (the code's number) and 61% in M-YQ0201-29.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min")